# core

> Shared data contracts, enumerations, and image-padding utilities used throughout
> the tiling-stitching pipeline.  
> All heavy lifting (tiling, stitching, model wrapping, ONNX export) lives in the
> sibling modules. This module provides the shared types and padding helpers
> (requires PyTorch for tensor ops).


In [ ]:
#| default_exp tiling_core

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
#| hide
from nbdev.showdoc import *

In [4]:
!which python

/home/ai_dsx.work/data/projects/goni/image_tiling_stiching_pt/.venv/bin/python


In [ ]:
#| export
from dataclasses import dataclass
from enum import Enum
from typing import Tuple

from loguru import logger
import torch
from torch import Tensor


In [ ]:
#| export
class PaddingMode(str, Enum):
    """How to pad an image so H and W are divisible by the tile stride.

    Values map directly to ``torch.nn.functional.pad`` mode strings.
    """
    REFLECT   = "reflect"    # Mirror-pad — avoids hard edges, best for natural images
    CONSTANT  = "constant"   # Zero-pad (or user-supplied value)
    REPLICATE = "replicate"  # Repeat edge pixels


In [ ]:
#| export
class MergeStrategy(str, Enum):
    """How overlapping tile predictions are blended in the stitcher.

    Pass the chosen strategy to :class:`~segmentation_tools_pt.stitching.TileMerger` /
    :class:`~segmentation_tools_pt.stitching.PatchAccumulator`.
    """
    GAUSSIAN_WEIGHTED    = "gaussian_weighted"     # Smooth centre-biased blend (recommended)
    AVERAGE              = "average"               # Equal-weight averaging
    CONFIDENCE_WEIGHTED  = "confidence_weighted"   # Weight by max-softmax score
    OVERWRITE            = "overwrite"             # Last tile wins (fastest, visible seams)


In [ ]:
#| export
@dataclass
class PatchCoords:
    """Pixel coordinates of a single tile in the full image space.

    All coordinates are **inclusive-start, exclusive-end** (Python slice convention).

    Attributes:
        y_start: Top row index.
        y_end:   Bottom row index (exclusive).
        x_start: Left column index.
        x_end:   Right column index (exclusive).

    Example::

        coords = PatchCoords(y_start=0, y_end=256, x_start=0, x_end=256)
        patch  = image[..., coords.y_start:coords.y_end, coords.x_start:coords.x_end]
    """
    y_start: int
    y_end:   int
    x_start: int
    x_end:   int

    @property
    def height(self) -> int:
        return self.y_end - self.y_start

    @property
    def width(self) -> int:
        return self.x_end - self.x_start

    def __repr__(self) -> str:  # pragma: no cover
        return (f"PatchCoords(y={self.y_start}:{self.y_end}, "
                f"x={self.x_start}:{self.x_end})")


In [ ]:
#| export
@dataclass
class TileConfig:
    """Static configuration for the tiling and stitching pipeline.
    """
    patch_size:   int # Side length (px) of each square tile fed to the model.
    overlap:      int          = 32 # Number of pixels that adjacent tiles share on each side.
    padding_mode: PaddingMode  = PaddingMode.REFLECT # How to extend image edges so H/W are divisible by the stride.
    pad_value:    float        = 0.0 # Fill value used when ``padding_mode == PaddingMode.CONSTANT``.

    def __post_init__(self) -> None:
        if not (0 <= self.overlap < self.patch_size):
            raise ValueError(
                f"overlap ({self.overlap}) must satisfy 0 <= overlap < patch_size ({self.patch_size})"
            )

    @property
    def stride(self) -> int:
        """Effective sliding-window step: ``patch_size - overlap``."""
        return self.patch_size - self.overlap


In [ ]:
#| export
@dataclass
class InferenceConfig:
    """Runtime configuration for model inference inside :class:`~segmentation_tools_pt.model_wrapper.TiledModel`.

    Args:
        inference_batch_size: Number of tiles sent to the model in a single forward pass.
                              Reduce if GPU memory is tight; increase for throughput on large images.
        device:               PyTorch device string, e.g. ``"cuda:0"`` or ``"cpu"``.
        merge_strategy:       How overlapping tile outputs are blended. See :class:`MergeStrategy`.
        confidence_threshold: Threshold used when ``merge_strategy == MergeStrategy.CONFIDENCE_WEIGHTED``.
        mask_output_index:    If the model returns a tuple/list, which index holds the segmentation mask tensor.
    """
    inference_batch_size: int           = 4
    device:               str           = "cpu"
    merge_strategy:       MergeStrategy = MergeStrategy.GAUSSIAN_WEIGHTED
    confidence_threshold: float         = 0.5
    mask_output_index:    int           = 0


In [ ]:
#| export
# Type alias: (pad_left, pad_top, pad_right, pad_bottom)
PadLTRB = Tuple[int, int, int, int]


def pad_to_divisible(
    image: Tensor,  # Shape (B, C, H, W) or (C, H, W)
    tile_cfg: TileConfig # Tiling configuration that supplies ``stride`` and ``padding_mode``.
    ) -> Tuple[Tensor, PadLTRB]:
    """Pad ``image`` so its H and W are divisible by ``tile_cfg.stride``.
    Example::

        cfg    = TileConfig(patch_size=256, overlap=32)
        padded, pad = pad_to_divisible(image, cfg)
        output = model(padded)
        result = unpad_image(output, pad)
    """
    stride = tile_cfg.stride
    h, w   = image.shape[-2], image.shape[-1]

    pad_bottom = (stride - h % stride) % stride
    pad_right  = (stride - w % stride) % stride

    if pad_bottom == 0 and pad_right == 0:
        return image, (0, 0, 0, 0)

    # torch.nn.functional.pad order: (left, right, top, bottom)
    padded = torch.nn.functional.pad(
        image,
        (0, pad_right, 0, pad_bottom),
        mode=tile_cfg.padding_mode.value,
        value=tile_cfg.pad_value if tile_cfg.padding_mode == PaddingMode.CONSTANT else 0.0,
    )
    logger.debug(
        "pad_to_divisible: ({},{}) -> ({},{})  pad_r={} pad_b={}",
        h, w, padded.shape[-2], padded.shape[-1], pad_right, pad_bottom,
    )
    return padded, (0, 0, pad_right, pad_bottom)


In [ ]:
#| export
def unpad_image(
    image: Tensor, # Shape (B, C, H_padded, W_padded) or (C, H_padded, W_padded)
    pad_ltrb: PadLTRB # (left, top, right, bottom) padding amounts returned by :func:`pad_to_divisible`.
    ) -> Tensor:
    """Remove the padding added by :func:`pad_to_divisible`.
    Returns:
        Tensor with the original spatial dimensions restored.
    """
    _, _, pad_right, pad_bottom = pad_ltrb
    h_end = image.shape[-2] - pad_bottom if pad_bottom > 0 else image.shape[-2]
    w_end = image.shape[-1] - pad_right  if pad_right  > 0 else image.shape[-1]
    return image[..., :h_end, :w_end]


In [ ]:
# test: pad_to_divisible / unpad_image round-trip
cfg = TileConfig(patch_size=256, overlap=32)
x = torch.rand(1, 3, 1000, 1500)
padded, pad_ltrb = pad_to_divisible(x, cfg)
assert padded.shape[-2] % cfg.stride == 0 and padded.shape[-1] % cfg.stride == 0
restored = unpad_image(padded, pad_ltrb)
assert restored.shape == x.shape


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()